## Initial Replication of Estimates from [1] 

[1] Combined Postmenopausal Hormone Therapy and Cardiovascular Disease: Toward Resolving the Discrepancy between Observational Studies and the Women’s Health Initiative Clinical Trial, Prentice et al., 2005

In [1]:
import pandas as pd 
import numpy as np 
import os 
import sys 
from tqdm import tqdm
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from scipy.stats import zscore

In [2]:

# read tables
dir_path = '/Users/zeshanmh/Documents/research/benchmarking-os/'
out   = pd.read_csv(os.path.join(dir_path, 'whi/data/data/main_study/csv/outc_adj_bio.csv'))
ct_fu = pd.read_csv(os.path.join(dir_path, 'whi/data/data/main_study/csv/adh_ht_pub.csv'))[['ID', 'ADHRATE', 'ENDDY', 'STARTDY', 'LOST', 'STOPHRT']] 
std_trt = pd.read_csv(os.path.join(dir_path, 'whi/data/data/main_study/csv/dem_ctos_bio.csv'))[['ID', 'HRTARM', 'OSFLAG']]



In [3]:
# List of outcomes     
glbl_list = ['CHD', 'BREAST', 'STROKE', 'PE', 'ENDMTRL', 'COLORECTAL', 'BKHIP', 'DEATH']    
other_list = ['PTCA', 'DVT']

In [4]:
# Get end of follow-up for CT patients 
# BTW, do we have to consider START-DAY? what about LOST for censoring?

# keep only those with ADHRATE not missing, and group by ID to get max ENDDY
# keep columns 'ID', 'ENDDY', and 'LOST'
# rename ENDDY to END_DY
# ct_end = ct_fu[ct_fu['ADHRATE'].notna()].groupby('ID')['ENDDY'].max().reset_index() 
# ct_end = ct_end.rename(columns={'ENDDY': 'END_DY'})
# ct_end

# ct_end = ct_fu[ct_fu['ADHRATE'].notna()][['ADHRATE','ID', 'ENDDY', 'LOST']].rename(columns={'ENDDY': 'END_DY'})
# ct_end = ct_fu[['ADHRATE','ID', 'ENDDY', 'LOST']].rename(columns={'ENDDY': 'END_DY'})
# ct_end = ct_end.query('ADHRATE != 0.')[['ID','END_DY','LOST']]
ct_end = ct_fu[ct_fu['ADHRATE'].notna()].groupby('ID')['ENDDY'].max().reset_index() 
ct_end = ct_end.rename(columns={'ENDDY': 'END_DY'})
ct_end

,ID,END_DY
0,500001,2190
1,500022,3287
2,500024,1095
3,500025,2190
4,500027,2921
...,...,...
27168,699951,2556
27169,699963,2556
27170,699974,1460
27171,699987,4017


In [5]:
ct_df = std_trt.drop_duplicates('ID')
ct_df = ct_df[ct_df['HRTARM'].isin(['E+P intervention', 'E+P control'])]
ct_df = ct_df.merge(ct_end, on='ID', how='left')
ct_df = ct_df.merge(out, on='ID', how='left')

# code variables HRTARM and OS 
ct_df['OS'] = 0 
ct_df['HRTARM'] = ct_df['HRTARM'].map({'E+P intervention': 1, 'E+P control': 0})

# print out first 10 rows
print(ct_df.shape)
print(ct_df[ct_df['HRTARM'] == 1].shape)
print(ct_df[ct_df['HRTARM'] == 0].shape)
ct_df.head(n=10)


(16608, 358)
(8506, 358)
(8102, 358)


,ID,HRTARM,OSFLAG,END_DY,ANGINA,ANGINADY,ANGINASRC,AANEUR,AANEURDY,AANEURSRC,...,DEATHDY,DEATHSRC,DEATHCAUSESRC,HYST,HYSTDY,HYSTSRC,ENDWHIDY,ENDEXT1DY,ENDFOLLOWDY,OS
0,642629,1,No,1460.0,0,NaN,NaN,0,NaN,NaN,...,NaN,NaN,NaN,1,4499.0,1.0,3480.0,5481.0,9083.0,0
1,568085,1,No,1825.0,0,NaN,NaN,0,NaN,NaN,...,7610.0,2.0,1.0,0,NaN,NaN,2725.0,4726.0,7610.0,0
2,568186,0,No,2555.0,0,NaN,NaN,0,NaN,NaN,...,NaN,NaN,NaN,0,NaN,NaN,3138.0,5139.0,8013.0,0
3,623255,1,No,1825.0,0,NaN,NaN,0,NaN,NaN,...,NaN,NaN,NaN,0,NaN,NaN,2500.0,4501.0,8015.0,0
4,537848,1,No,2555.0,0,NaN,NaN,0,NaN,NaN,...,NaN,NaN,NaN,0,NaN,NaN,3263.0,5264.0,8992.0,0
5,668539,1,No,2556.0,0,NaN,NaN,0,NaN,NaN,...,NaN,NaN,NaN,0,NaN,NaN,3486.0,5487.0,5487.0,0
6,660370,1,No,1095.0,0,NaN,NaN,0,NaN,NaN,...,827.0,1.0,1.0,0,NaN,NaN,827.0,827.0,827.0,0
7,551999,1,No,1094.0,0,NaN,NaN,0,NaN,NaN,...,3926.0,1.0,1.0,0,NaN,NaN,3012.0,3926.0,3926.0,0
8,682433,0,No,1095.0,0,NaN,NaN,0,NaN,NaN,...,NaN,NaN,NaN,1,834.0,0.0,2521.0,4522.0,8021.0,0
9,605069,1,No,1460.0,0,NaN,NaN,0,NaN,NaN,...,NaN,NaN,NaN,0,NaN,NaN,2681.0,4682.0,8375.0,0


In [6]:
# process outcomes 
for i in glbl_list + other_list: 
    ct_df[i+'_E']  = ((ct_df[i] == 1) & (ct_df[i+'DY'] <= ct_df['END_DY'])).astype(int)
    ct_df[i+'_DY'] = np.where(ct_df[i+'_E'] == 1, ct_df[i+'DY'], ct_df['END_DY'])
    ct_df[i+'_EDY'] = np.where(ct_df[i+'_E'] == 1, ct_df[i+'DY'], np.nan) 

# Global index
ct_df['GLBL_E'] = (ct_df[[j+'_E' for j in glbl_list]].sum(axis=1) > 0).astype(int)
ct_df['GLBL_DY'] = np.where(ct_df['GLBL_E'] == 1,
                            ct_df[[j+'_EDY' for j in glbl_list]].min(axis=1),
                            ct_df[[j+'_DY' for j in glbl_list]].min(axis=1))

# Select needed columns
ct_df = ct_df[['ID', 'OS', 'HRTARM'] + 
                [j+'_E' for j in glbl_list + other_list + ['GLBL']] + 
                [j+'_DY' for j in glbl_list + other_list + ['GLBL']]]


In [7]:
ct_df.query('HRTARM == 0 & CHD_E == 1')

,ID,OS,HRTARM,CHD_E,BREAST_E,STROKE_E,PE_E,ENDMTRL_E,COLORECTAL_E,BKHIP_E,...,BREAST_DY,STROKE_DY,PE_DY,ENDMTRL_DY,COLORECTAL_DY,BKHIP_DY,DEATH_DY,PTCA_DY,DVT_DY,GLBL_DY
21,603833,0,0,1,0,0,1,0,0,0,...,729.0,729.0,538.0,729.0,729.0,729.0,729.0,729.0,729.0,508.0
31,592726,0,0,1,0,0,0,0,0,0,...,2921.0,2921.0,2921.0,2921.0,2921.0,2921.0,2921.0,2921.0,2921.0,2036.0
78,602131,0,0,1,0,0,0,0,0,0,...,2191.0,2191.0,2191.0,2191.0,2191.0,2191.0,2191.0,2191.0,2191.0,2180.0
275,592897,0,0,1,0,0,0,0,0,0,...,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2097.0,2190.0,2190.0,2097.0
292,520105,0,0,1,0,0,0,0,0,0,...,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2102.0,2190.0,2190.0,2102.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16066,614132,0,0,1,0,0,0,0,0,0,...,1826.0,1826.0,1826.0,1826.0,1826.0,1826.0,1593.0,1826.0,1826.0,1593.0
16203,678820,0,0,1,0,0,0,0,0,0,...,2555.0,2555.0,2555.0,2555.0,2555.0,2555.0,2555.0,912.0,2555.0,912.0
16234,664063,0,0,1,0,0,0,0,0,0,...,2556.0,2556.0,2556.0,2556.0,2556.0,2556.0,2556.0,1342.0,2556.0,1342.0
16249,597604,0,0,1,0,0,0,0,0,0,...,2191.0,2191.0,2191.0,2191.0,2191.0,2191.0,1913.0,2191.0,2191.0,1913.0


In [8]:
dir_path = '/Users/zeshanmh/Documents/research/benchmarking-os/'
hyst    = pd.read_csv(os.path.join(dir_path, 'whi/data/data/main_study/csv/f2_ctos_bio.csv'))[['ID','HYST']]
pre_hrt  = pd.read_csv(os.path.join(dir_path, 'whi/data/data/main_study/csv/f43_ctos_bio.csv'))[['ID', 'TOTESTAT','TOTPSTAT']]
post_hrt = pd.read_csv(os.path.join(dir_path, 'whi/data/data/main_study/csv/f48_av1_os_pub.csv'))[['ID','ELSTYR','PLSTYR','HRTCMBP']]

/var/folders/t9/9775q6dn21l67f71h7t7xj0h0000gn/T/ipykernel_68399/907246142.py:2: DtypeWarning: Columns (39,42,57,59,66) have mixed types. Specify dtype option on import or set low_memory=False.
  hyst    = pd.read_csv(os.path.join(dir_path, 'whi/data/data/main_study/csv/f2_ctos_bio.csv'))[['ID','HYST']]


In [9]:
out.columns

Index(['ID', 'ANGINA', 'ANGINADY', 'ANGINASRC', 'AANEUR', 'AANEURDY',
       'AANEURSRC', 'ADISSECT', 'ADISSECTDY', 'ADISSECTSRC',
       ...
       'DEATHCAUSE', 'DEATHDY', 'DEATHSRC', 'DEATHCAUSESRC', 'HYST', 'HYSTDY',
       'HYSTSRC', 'ENDWHIDY', 'ENDEXT1DY', 'ENDFOLLOWDY'],
      dtype='object', length=354)

In [40]:
os_df_temp = std_trt.drop_duplicates('ID')
os_df_temp = os_df_temp[os_df_temp['OSFLAG'] == 'Yes']
os_df_temp = os_df_temp.merge(hyst, on='ID', how='left')
os_df_temp = os_df_temp.merge(pre_hrt, on='ID', how='left')
# os_df_temp = os_df_temp[os_df_temp['HYST'] == 'Yes']
print(os_df_temp.columns)
os_df_temp = os_df_temp.query('HYST.astype("string") == "Yes" | TOTESTAT.astype("string") == "Current user"')

# os_df_temp = os_df_temp[os_df_temp['TOTESTAT'] == 'Current user']
unc_hf = pd.read_csv(os.path.join(dir_path, 'whi/data/data/main_study/csv/unc_hf_bio.csv'))[['ID', 'CHDYRHX', 'CHDEVERHX']]
os_df_temp = os_df_temp.merge(unc_hf, on='ID', how='left') 
print(os_df_temp['CHDYRHX'].value_counts())
print(os_df_temp['CHDEVERHX'].value_counts())
print(os_df_temp['TOTPSTAT'].value_counts())

Index(['ID', 'HRTARM', 'OSFLAG', 'HYST', 'TOTESTAT', 'TOTPSTAT'], dtype='object')
CHDYRHX
0.0    692
1.0     61
Name: count, dtype: int64
CHDEVERHX
0.0    442
1.0    250
Name: count, dtype: int64
TOTPSTAT
Never used      36693
Past user        3353
Current user      793
Name: count, dtype: int64


In [10]:
# construct os_df 
os_df = std_trt.drop_duplicates('ID')
os_df = os_df[os_df['OSFLAG'] == 'Yes']
os_df = os_df.merge(hyst, on='ID', how='left')
os_df = os_df[os_df['HYST'] == 'No']
os_df = os_df.merge(pre_hrt, on='ID', how='left')
print(os_df['TOTESTAT'].value_counts())
os_df = os_df[os_df['TOTESTAT'].isin(['Never used', 'Past user'])]
os_df = os_df.merge(post_hrt, on='ID', how='left')
os_df = os_df.merge(out, on='ID', how='left')

# 35551 (control) + 17503 (intervention) = 53054
print(os_df[os_df['TOTPSTAT'].isin(['Current user'])].shape)
print(os_df[os_df['TOTPSTAT'].isin(['Never used', 'Past user'])].shape)
os_df = os_df[os_df['TOTPSTAT'].isin(['Never used', 'Past user','Current user'])]
os_df['OS'] = 1
os_df['HRTARM'] = os_df['TOTPSTAT'].map({'Current user': 1, 'Never used': 0, 'Past user': 0})

# os_end_day = None
os_end_day = 6*365
os_df['END_DY'] = os_end_day if os_end_day is not None else os_df['ENDFOLLOWDY']
# os_df['END_DY'] = os_df.apply(lambda x: x['DEATHDY'] if x['DEATHDY'] < os_end_day else os_end_day, axis=1)

# Process outcomes (same as CT)
for i in glbl_list + other_list:
    os_df[i+'_E'] = ((os_df[i] == 1) & (os_df[i+'DY'] <= os_df['END_DY'])).astype(int)
    os_df[i+'_DY'] = np.where(os_df[i+'_E'] == 1, os_df[i+'DY'], os_df['END_DY'])
    os_df[i+'_EDY'] = np.where(os_df[i+'_E'] == 1, os_df[i+'_DY'], np.nan)

# Global index
os_df['GLBL_E'] = (os_df[[j+'_E' for j in glbl_list]].sum(axis=1) > 0).astype(int)
os_df['GLBL_DY'] = np.where(os_df['GLBL_E'] == 1,
                            os_df[[j+'_EDY' for j in glbl_list]].min(axis=1),
                            os_df[[j+'_DY' for j in glbl_list]].min(axis=1))

# Select needed columns
os_df = os_df[['ID', 'OS', 'HRTARM'] + 
                [j+'_E' for j in glbl_list + other_list + ['GLBL']] + 
                [j+'_DY' for j in glbl_list + other_list + ['GLBL']]]

os_df


# Below is what you would do for proper trial emulation LOL
# conditions = [
#     (((os_df['ELSTYR'] == 'Yes') & (os_df['PLSTYR'] == 'Yes')) | (os_df['HRTCMBP'] == 'Yes')),
#     ((os_df['ELSTYR'] == 'No') & (os_df['PLSTYR'] == 'No')),
#     (((os_df['ELSTYR'] == 'Yes') & (os_df['PLSTYR'] == 'No')) | ((os_df['ELSTYR'] == 'No') & (os_df['PLSTYR'] == 'Yes')))
# ]
# choices = [1, 0, -1]
# os_df['HRTGRP'] = np.select(conditions, choices, default=-2)
# os_df = os_df[os_df['HRTGRP'] != -2]
# os_df['HRTARM'] = (os_df['HRTGRP'] == 1).astype(int)


TOTESTAT
Never used      47832
Past user        5244
Current user     1343
Name: count, dtype: int64
(17509, 362)
(35539, 362)


,ID,OS,HRTARM,CHD_E,BREAST_E,STROKE_E,PE_E,ENDMTRL_E,COLORECTAL_E,BKHIP_E,...,BREAST_DY,STROKE_DY,PE_DY,ENDMTRL_DY,COLORECTAL_DY,BKHIP_DY,DEATH_DY,PTCA_DY,DVT_DY,GLBL_DY
0,591800,1,0,0,0,0,0,0,0,0,...,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0
1,548198,1,0,0,0,0,0,0,0,0,...,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0
2,670866,1,1,0,0,0,0,0,0,0,...,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0
3,665473,1,1,0,0,0,0,0,0,0,...,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0
4,508135,1,0,0,0,0,0,0,0,0,...,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53071,641028,1,0,0,0,0,0,0,0,0,...,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0
53072,687963,1,0,0,1,0,0,0,0,0,...,203.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,203.0
53073,681994,1,1,0,0,0,0,0,0,0,...,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0
53074,575472,1,0,0,0,0,0,0,0,0,...,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0


In [11]:
# check outcome of breast cancer and CHD in both ct and os
print('Clinical Trial')
print(f'No. of women in control arm: {ct_df[ct_df["HRTARM"] == 0].shape[0]}')
print(f'No. of women in intervention arm: {ct_df[ct_df["HRTARM"] == 1].shape[0]}')
print(f'No. of CHD events in control arm: {ct_df.query("HRTARM == 0 & CHD_E == 1").shape[0]}')
print(f'No. of CHD events in intervention arm: {ct_df.query("HRTARM == 1 & CHD_E == 1").shape[0]}')
print(f'No. of breast cancer events in control arm: {ct_df.query("HRTARM == 0 & BREAST_E == 1").shape[0]}')
print(f'No. of breast cancer events in intervention arm: {ct_df.query("HRTARM == 1 & BREAST_E == 1").shape[0]}')
print(f'No. of stroke events in control arm: {ct_df.query("HRTARM == 0 & STROKE_E == 1").shape[0]}')
print(f'No. of stroke events in intervention arm: {ct_df.query("HRTARM == 1 & STROKE_E == 1").shape[0]}')
print()
print()

print('Observational Study')
print(f'No. of women in control arm: {os_df[os_df["HRTARM"] == 0].shape[0]}')
print(f'No. of women in intervention arm: {os_df[os_df["HRTARM"] == 1].shape[0]}')
print(f'No. of CHD events in control arm: {os_df.query("HRTARM == 0 & CHD_E == 1").shape[0]}')
print(f'No. of CHD events in intervention arm: {os_df.query("HRTARM == 1 & CHD_E == 1").shape[0]}')
print(f'No. of breast cancer events in control arm: {os_df.query("HRTARM == 0 & BREAST_E == 1").shape[0]}')
print(f'No. of breast cancer events in intervention arm: {os_df.query("HRTARM == 1 & BREAST_E == 1").shape[0]}')
print(f'No. of stroke events in control arm: {os_df.query("HRTARM == 0 & STROKE_E == 1").shape[0]}')
print(f'No. of stroke events in intervention arm: {os_df.query("HRTARM == 1 & STROKE_E == 1").shape[0]}')


Clinical Trial
No. of women in control arm: 8102
No. of women in intervention arm: 8506
No. of CHD events in control arm: 129
No. of CHD events in intervention arm: 169
No. of breast cancer events in control arm: 169
No. of breast cancer events in intervention arm: 231
No. of stroke events in control arm: 88
No. of stroke events in intervention arm: 124


Observational Study
No. of women in control arm: 35539
No. of women in intervention arm: 17509
No. of CHD events in control arm: 703
No. of CHD events in intervention arm: 184
No. of breast cancer events in control arm: 1085
No. of breast cancer events in intervention arm: 834
No. of stroke events in control arm: 542
No. of stroke events in intervention arm: 142


In [12]:
# Combine CT and OS data
ctos_df = pd.concat([ct_df, os_df], ignore_index=True)


In [13]:
ctos_df

,ID,OS,HRTARM,CHD_E,BREAST_E,STROKE_E,PE_E,ENDMTRL_E,COLORECTAL_E,BKHIP_E,...,BREAST_DY,STROKE_DY,PE_DY,ENDMTRL_DY,COLORECTAL_DY,BKHIP_DY,DEATH_DY,PTCA_DY,DVT_DY,GLBL_DY
0,642629,0,1,0,0,0,1,0,0,0,...,1460.0,1460.0,935.0,1460.0,1460.0,1460.0,1460.0,1460.0,936.0,935.0
1,568085,0,1,0,0,0,0,0,0,0,...,1825.0,1825.0,1825.0,1825.0,1825.0,1825.0,1825.0,1825.0,1825.0,1825.0
2,568186,0,0,0,0,0,0,0,0,0,...,2555.0,2555.0,2555.0,2555.0,2555.0,2555.0,2555.0,2555.0,2555.0,2555.0
3,623255,0,1,0,0,0,0,0,0,0,...,1825.0,1825.0,1825.0,1825.0,1825.0,1825.0,1825.0,1825.0,1825.0,1825.0
4,537848,0,1,0,0,0,0,0,0,0,...,2555.0,2555.0,2555.0,2555.0,2555.0,2555.0,2555.0,2555.0,2555.0,2555.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69651,641028,1,0,0,0,0,0,0,0,0,...,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0
69652,687963,1,0,0,1,0,0,0,0,0,...,203.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,203.0
69653,681994,1,1,0,0,0,0,0,0,0,...,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0
69654,575472,1,0,0,0,0,0,0,0,0,...,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0


# Analysis

In [14]:
import pandas.api.types as ptypes

ctos_temp = ctos_df.copy()
# Dictionary to specify which features are categorical
categorical_features = {
    'dem_ctos_bio.csv': {'AGE': False, 'ETHNIC': True, 'EDUC': True}, 
    'f80_ctos_bio.csv': {'BMI': False}, 
    'f34_ctos_bio.csv': {'SMOKING': True}, 
    'f31_ctos_bio.csv': {'MENO': False}, 
    'f151_ctos_bio.csv': {'PHYSFUN': False}    
}

new_feature_dict = { 
    'dem_ctos_bio.csv': ['AGE', 'ETHNIC_White', \
                         'EDUC_Some post-graduate or professional', \
                         'EDUC_Some college or Associate Degree'],
    'f80_ctos_bio.csv': ['BMI'],
    'f34_ctos_bio.csv': ['SMOKING_Past Smoker', 'SMOKING_Current Smoker'],
    'f31_ctos_bio.csv': ['MENO'],
    'f151_ctos_bio.csv': ['PHYSFUN']
}

# dfs = []  # Store all dataframes to concatenate later
new_dir_path = dir_path + 'whi/data/data/main_study/csv'

for filename, f_dict in categorical_features.items():
    # Read the data
    df = pd.read_csv(os.path.join(new_dir_path, filename))
    if filename == 'f80_ctos_bio.csv': 
        df = df.query('F80VTYP == "Screening"')
    elif filename == 'f151_ctos_bio.csv': 
        idx = df.groupby('ID')['F151DAYS'].idxmin().reset_index(drop=True)
        df = df.loc[idx, :].reset_index(drop=True)[['ID','PHYSFUN']]
    # Select needed columns
    features = list(f_dict.keys())
    df = df[['ID'] + features]
    
    # Separate ID column
    id_col = df['ID']
    print(f"Processed {filename}")
    print(df.shape)

    orig_cols = ctos_temp.columns.tolist()
    ctos_temp = ctos_temp.merge(df, on='ID', how='left')

    # Handle continuous and categorical features separately
    cont_features = [f for f in features if not f_dict[f]]
    cat_features = [f for f in features if f_dict[f]]
    
    # Handle continuous features
    if cont_features:
        cont_imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
        ctos_temp[cont_features] = cont_imputer.fit_transform(ctos_temp[cont_features])
    
    # Handle categorical features
    if cat_features:
        cat_imputer = SimpleImputer(missing_values=np.nan, strategy='most_frequent')
        ctos_temp[cat_features] = cat_imputer.fit_transform(ctos_temp[cat_features])
        
        # One-hot encode categorical features
        ctos_temp = pd.get_dummies(ctos_temp, columns=cat_features, prefix=cat_features)

    if filename == 'dem_ctos_bio.csv': 
        ctos_temp = ctos_temp.rename(columns={'ETHNIC_White (not of Hispanic origin)': 'ETHNIC_White'})

    ctos_temp = ctos_temp[orig_cols + new_feature_dict[filename]]

ctos_temp = ctos_temp.astype({col: int for col in ctos_temp.select_dtypes(include='bool').columns})
display(ctos_temp)    


Processed dem_ctos_bio.csv
(161808, 4)
Processed f80_ctos_bio.csv
(161771, 2)
Processed f34_ctos_bio.csv
(161625, 2)


/var/folders/t9/9775q6dn21l67f71h7t7xj0h0000gn/T/ipykernel_68399/2535602248.py:28: DtypeWarning: Columns (20,22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(new_dir_path, filename))


Processed f31_ctos_bio.csv
(161705, 2)
Processed f151_ctos_bio.csv
(113491, 2)


,ID,OS,HRTARM,CHD_E,BREAST_E,STROKE_E,PE_E,ENDMTRL_E,COLORECTAL_E,BKHIP_E,...,GLBL_DY,AGE,ETHNIC_White,EDUC_Some post-graduate or professional,EDUC_Some college or Associate Degree,BMI,SMOKING_Past Smoker,SMOKING_Current Smoker,MENO,PHYSFUN
0,642629,0,1,0,0,0,1,0,0,0,...,935.0,64.0,1,0,0,29.19411,0,0,54.0,65.000000
1,568085,0,1,0,0,0,0,0,0,0,...,1825.0,62.0,0,0,0,19.55943,0,1,51.0,90.000000
2,568186,0,0,0,0,0,0,0,0,0,...,2555.0,62.0,1,0,1,30.44928,1,0,44.0,50.000000
3,623255,0,1,0,0,0,0,0,0,0,...,1825.0,60.0,1,1,0,28.54828,0,0,54.0,78.137062
4,537848,0,1,0,0,0,0,0,0,0,...,2555.0,54.0,1,0,1,40.32766,1,0,54.0,65.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69651,641028,1,0,0,0,0,0,0,0,0,...,2190.0,69.0,1,1,0,25.50301,0,0,54.0,90.000000
69652,687963,1,0,0,1,0,0,0,0,0,...,203.0,70.0,1,0,1,23.11101,1,0,55.0,60.000000
69653,681994,1,1,0,0,0,0,0,0,0,...,2190.0,53.0,1,0,0,26.76123,0,0,48.0,95.000000
69654,575472,1,0,0,0,0,0,0,0,0,...,2190.0,75.0,1,0,0,28.07091,1,0,50.0,78.137062


In [15]:
# hazard ratios for stroke, breast cancer, and CHD in clinical trial vs observational study 

## CT 
ct_df = ctos_temp.query('OS == 0')
ct_df_sub = ct_df[['ID','HRTARM', 'STROKE_E', 'BREAST_E', 'CHD_E','STROKE_DY', 'BREAST_DY', 'CHD_DY']]
ct_df_chd = ct_df[['HRTARM', 'CHD_E', 'CHD_DY']]
ct_df_chd = ct_df_chd[ct_df_chd['CHD_DY'].notna()]

ct_df_stroke = ct_df[['HRTARM', 'STROKE_E', 'STROKE_DY']]
ct_df_stroke = ct_df_stroke[ct_df_stroke['STROKE_DY'].notna()]

ct_df_breast = ct_df[['HRTARM', 'BREAST_E', 'BREAST_DY']]
ct_df_breast = ct_df_breast[ct_df_breast['BREAST_DY'].notna()]

from lifelines import CoxPHFitter

def get_hr(df, Y, E, event_name, HR_cov='HRTARM', study_type='Clinical Trial'): 
    cph = CoxPHFitter()
    cph.fit(df, duration_col=Y, event_col=E)
    cph.print_summary()
    cHR = cph.hazard_ratios_[HR_cov]
    cis = cph.confidence_intervals_
    lower = np.exp(cis['95% lower-bound'][HR_cov])
    upper = np.exp(cis['95% upper-bound'][HR_cov])
    print(f'Hazard ratio for {event_name} in {study_type}: {np.round(cHR, 2)} (95% CI: {np.round(lower, 2)}, {np.round(upper, 2)})')

get_hr(ct_df_chd, 'CHD_DY', 'CHD_E', 'CHD')
get_hr(ct_df_stroke, 'STROKE_DY', 'STROKE_E', 'Stroke')
get_hr(ct_df_breast, 'BREAST_DY', 'BREAST_E', 'Breast Cancer')


<lifelines.CoxPHFitter: fitted with 16516 total observations, 16218 right-censored observations>
             duration col = 'CHD_DY'
                event col = 'CHD_E'
      baseline estimation = breslow
   number of observations = 16516
number of events observed = 298
   partial log-likelihood = -2787.04
         time fit was run = 2025-01-09 18:09:33 UTC

---
           coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                  
HRTARM     0.24      1.28      0.12            0.01            0.47                1.01                1.61

           cmp to    z    p  -log2(p)
covariate                            
HRTARM       0.00 2.09 0.04      4.76
---
Concordance = 0.54
Partial AIC = 5576.08
log-likelihood ratio test = 4.39 on 1 df
-log2(p) of ll-ratio test = 4.79

Hazard ratio for CHD in Clinical Trial: 1.28 (95% CI: 1.01, 1.61)


<lifelines.CoxPHFitter: fitted with 16516 total observations, 16304 right-censored observations>
             duration col = 'STROKE_DY'
                event col = 'STROKE_E'
      baseline estimation = breslow
   number of observations = 16516
number of events observed = 212
   partial log-likelihood = -1970.95
         time fit was run = 2025-01-09 18:09:34 UTC

---
           coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                  
HRTARM     0.32      1.37      0.14            0.04            0.59                1.04                1.80

           cmp to    z    p  -log2(p)
covariate                            
HRTARM       0.00 2.27 0.02      5.41
---
Concordance = 0.53
Partial AIC = 3943.90
log-likelihood ratio test = 5.21 on 1 df
-log2(p) of ll-ratio test = 5.47

Hazard ratio for Stroke in Clinical Trial: 1.37 (95% CI: 1.04, 1.8)


<lifelines.CoxPHFitter: fitted with 16516 total observations, 16116 right-censored observations>
             duration col = 'BREAST_DY'
                event col = 'BREAST_E'
      baseline estimation = breslow
   number of observations = 16516
number of events observed = 400
   partial log-likelihood = -3713.55
         time fit was run = 2025-01-09 18:09:34 UTC

---
           coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                  
HRTARM     0.29      1.34      0.10            0.09            0.49                1.09                1.63

           cmp to    z      p  -log2(p)
covariate                              
HRTARM       0.00 2.86 <0.005      7.86
---
Concordance = 0.52
Partial AIC = 7429.11
log-likelihood ratio test = 8.25 on 1 df
-log2(p) of ll-ratio test = 7.94

Hazard ratio for Breast Cancer in Clinical Trial: 1.34 (95% CI: 1.09, 1.63)


In [16]:
# OS 
os_df = ctos_temp.query('OS == 1')
features = ['AGE','ETHNIC_White', 'EDUC_Some post-graduate or professional', \
            'EDUC_Some college or Associate Degree', 'BMI', 'SMOKING_Past Smoker', \
            'SMOKING_Current Smoker', 'MENO', 'PHYSFUN']
treatment = ['HRTARM']
events = ['CHD_E', 'CHD_DY']
event_name = 'CHD'

os_df_sub = os_df[features + treatment + events]
os_df_sub = os_df_sub[os_df_sub[events[1]].notna()]

get_hr(os_df_sub, events[1], events[0], event_name, HR_cov='HRTARM', study_type='Observational Study')




<lifelines.CoxPHFitter: fitted with 53048 total observations, 52161 right-censored observations>
             duration col = 'CHD_DY'
                event col = 'CHD_E'
      baseline estimation = breslow
   number of observations = 53048
number of events observed = 887
   partial log-likelihood = -9361.80
         time fit was run = 2025-01-09 18:09:52 UTC

---
                                         coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                                                
AGE                                      0.10      1.10      0.01            0.09            0.11                1.09                1.11
ETHNIC_White                             0.02      1.02      0.10           -0.17            0.21                0.84                1.24
EDUC_Some post-graduate or professional -0.26      0.77      0.11           -0.48           -0.03                0.62                0.97
EDUC_Some college or Associate Degree   -0.21      0.81      0.08           -0.37           -0.06                0.69                0.95
BMI                                      0.04      1.04      0.00            0.03            0.05                1.03                1.06
SMOKING_Past Smoker                      0.26      1.30      0.07            0.13            0.40                1.13                1.49
SMOKING_Current Smoker                   0.63      1.88      0.13            0.38            0.89                1.46                2.42
MENO                                    -0.02      0.98      0.01           -0.03           -0.01                0.97                0.99
PHYSFUN                                 -0.00      1.00      0.00           -0.01           -0.00                0.99                1.00
HRTARM                                  -0.14      0.87      0.09           -0.31            0.03                0.74                1.03

                                         cmp to     z      p  -log2(p)
covariate                                                             
AGE                                        0.00 18.46 <0.005    250.36
ETHNIC_White                               0.00  0.21   0.84      0.26
EDUC_Some post-graduate or professional    0.00 -2.25   0.02      5.34
EDUC_Some college or Associate Degree      0.00 -2.64   0.01      6.91
BMI                                        0.00  8.88 <0.005     60.32
SMOKING_Past Smoker                        0.00  3.74 <0.005     12.41
SMOKING_Current Smoker                     0.00  4.85 <0.005     19.64
MENO                                       0.00 -2.77   0.01      7.47
PHYSFUN                                    0.00 -2.28   0.02      5.49
HRTARM                                     0.00 -1.57   0.12      3.11
---
Concordance = 0.72
Partial AIC = 18743.60
log-likelihood ratio test = 560.77 on 10 df
-log2(p) of ll-ratio test = 376.55

Hazard ratio for CHD in Observational Study: 0.87 (95% CI: 0.74, 1.03)


# Fix with including the confounder (doesn't work)

In [17]:
os_df = ctos_temp.query('OS == 1')
pre_hrt_time  = pd.read_csv(os.path.join(dir_path, \
            'whi/data/data/main_study/csv/f43_ctos_bio.csv'))[['ID', 'TOTPTIME']]
os_df = os_df.merge(pre_hrt_time, on='ID', how='left') 
os_df['TAU'] = os_df.apply(lambda x: x['TOTPTIME'] if x['HRTARM'] == 1 else 0, axis=1)
# remove TOTPTIME as column from os_df inplace 
os_df.drop(columns=['TOTPTIME'], inplace=True)




In [45]:
features = ['AGE','ETHNIC_White', 'EDUC_Some post-graduate or professional', \
            'EDUC_Some college or Associate Degree', 'BMI', 'SMOKING_Past Smoker', \
            'SMOKING_Current Smoker', 'MENO', 'PHYSFUN', 'TAU']
treatment = ['HRTARM']
events = ['CHD_E', 'CHD_DY']
event_name = 'CHD'

os_df_sub = os_df[features + treatment + events]
os_df_sub = os_df_sub[os_df_sub[events[1]].notna()]

get_hr(os_df_sub, events[1], events[0], event_name, HR_cov='HRTARM', study_type='Observational Study')

<lifelines.CoxPHFitter: fitted with 53048 total observations, 52161 right-censored observations>
             duration col = 'CHD_DY'
                event col = 'CHD_E'
      baseline estimation = breslow
   number of observations = 53048
number of events observed = 887
   partial log-likelihood = -9361.44
         time fit was run = 2024-12-02 01:02:39 UTC

---
                                         coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                                                
AGE                                      0.10      1.10      0.01            0.09            0.11                1.09                1.11
ETHNIC_White                             0.02      1.02      0.10           -0.17            0.21                0.84                1.24
EDUC_Some post-graduate or professional -0.26      0.77      0.11           -0.48           -0.03                0.62                0.97
EDUC_Some college or Associate Degree   -0.21      0.81      0.08           -0.37           -0.06                0.69                0.95
BMI                                      0.04      1.04      0.00            0.03            0.05                1.03                1.06
SMOKING_Past Smoker                      0.26      1.30      0.07            0.12            0.40                1.13                1.49
SMOKING_Current Smoker                   0.63      1.88      0.13            0.38            0.89                1.46                2.42
MENO                                    -0.02      0.98      0.01           -0.03           -0.00                0.97                1.00
PHYSFUN                                 -0.00      1.00      0.00           -0.01           -0.00                0.99                1.00
TAU                                      0.01      1.01      0.01           -0.01            0.03                0.99                1.03
HRTARM                                  -0.22      0.80      0.13           -0.48            0.04                0.62                1.04

                                         cmp to     z      p  -log2(p)
covariate                                                             
AGE                                        0.00 18.18 <0.005    242.83
ETHNIC_White                               0.00  0.21   0.84      0.26
EDUC_Some post-graduate or professional    0.00 -2.25   0.02      5.36
EDUC_Some college or Associate Degree      0.00 -2.64   0.01      6.93
BMI                                        0.00  8.90 <0.005     60.58
SMOKING_Past Smoker                        0.00  3.73 <0.005     12.35
SMOKING_Current Smoker                     0.00  4.85 <0.005     19.64
MENO                                       0.00 -2.63   0.01      6.85
PHYSFUN                                    0.00 -2.28   0.02      5.46
TAU                                        0.00  0.86   0.39      1.36
HRTARM                                     0.00 -1.67   0.10      3.39
---
Concordance = 0.72
Partial AIC = 18744.87
log-likelihood ratio test = 561.49 on 11 df
-log2(p) of ll-ratio test = 374.12

Hazard ratio for CHD in Observational Study: 0.8 (95% CI: 0.62, 1.04)


In [ ]:
ctr_df = os_df.query('HRTARM == 0')
trt_df = os_df.query('HRTARM == 1')
trt_df_l2 = trt_df.query('TAU < 2')
trt_df_l2_5 = trt_df.query('(TAU >= 2 & TAU < 5)')
trt_df_g5 = trt_df.query('TAU >= 5')
 
# randomly split the control into 3 groups 
n = ctr_df.shape[0]
idxs = np.arange(n)
np.random.shuffle(idxs)
idxs_groups = np.array_split(idxs, 3)
crt_df_1 = ctr_df.loc[idxs_groups[0], :]
crt_df_2 = ctr_df.loc[idxs_groups[1], :]
crt_df_3 = ctr_df.loc[idxs_groups[2], :]

#concatenate trt and ctrl groups 
os_df_l2 = pd.concat([trt_df_l2, crt_df_1], ignore_index=True)
os_df_l2_5 = pd.concat([trt_df_l2_5, crt_df_2], ignore_index=True)
os_df_g5 = pd.concat([trt_df_g5, crt_df_3], ignore_index=True)

features = ['AGE','ETHNIC_White', 'EDUC_Some post-graduate or professional', \
            'EDUC_Some college or Associate Degree', 'BMI', 'SMOKING_Past Smoker', \
            'SMOKING_Current Smoker', 'MENO', 'PHYSFUN']
treatment = ['HRTARM']
events = ['CHD_E', 'CHD_DY']
event_name = 'CHD'

print(os_df.query('CHD_E == 1 & HRTARM == 0').shape)

os_df_sub = os_df_l2[features + treatment + events]
os_df_sub = os_df_sub[os_df_sub[events[1]].notna()]
print(os_df_sub.query('CHD_E == 1 & HRTARM == 0').shape)
get_hr(os_df_sub, events[1], events[0], event_name, HR_cov='HRTARM', study_type='Observational Study')

os_df_sub = os_df_l2_5[features + treatment + events]
os_df_sub = os_df_sub[os_df_sub[events[1]].notna()]
# print(os_df_sub.query('CHD_E == 1 & HRTARM == 0').shape)
get_hr(os_df_sub, events[1], events[0], event_name, HR_cov='HRTARM', study_type='Observational Study')

os_df_sub = os_df_g5[features + treatment + events]
os_df_sub = os_df_sub[os_df_sub[events[1]].notna()]
print(os_df_sub.query('CHD_E == 1 & HRTARM == 1').shape)
get_hr(os_df_sub, events[1], events[0], event_name, HR_cov='HRTARM', study_type='Observational Study')

In [58]:
# instead, try stratifying by TAU
os_df_l2 = os_df.query('TAU < 2')
os_df_l2_5 = os_df.query('(TAU >= 2 & TAU < 5) | HRTARM == 0') 
os_df_g5 = os_df.query('TAU >= 5 | HRTARM == 0')

features = ['AGE','ETHNIC_White', 'EDUC_Some post-graduate or professional', \
            'EDUC_Some college or Associate Degree', 'BMI', 'SMOKING_Past Smoker', \
            'SMOKING_Current Smoker', 'MENO', 'PHYSFUN']
treatment = ['HRTARM']
events = ['CHD_E', 'CHD_DY']
event_name = 'CHD'

print(os_df.query('CHD_E == 1 & HRTARM == 0').shape)

os_df_sub = os_df_l2[features + treatment + events]
os_df_sub = os_df_sub[os_df_sub[events[1]].notna()]
print(os_df_sub.query('CHD_E == 1 & HRTARM == 0').shape)
get_hr(os_df_sub, events[1], events[0], event_name, HR_cov='HRTARM', study_type='Observational Study')

os_df_sub = os_df_l2_5[features + treatment + events]
os_df_sub = os_df_sub[os_df_sub[events[1]].notna()]
# print(os_df_sub.query('CHD_E == 1 & HRTARM == 0').shape)
get_hr(os_df_sub, events[1], events[0], event_name, HR_cov='HRTARM', study_type='Observational Study')

os_df_sub = os_df_g5[features + treatment + events]
os_df_sub = os_df_sub[os_df_sub[events[1]].notna()]
print(os_df_sub.query('CHD_E == 1 & HRTARM == 1').shape)
get_hr(os_df_sub, events[1], events[0], event_name, HR_cov='HRTARM', study_type='Observational Study')

(703, 35)
(703, 12)


<lifelines.CoxPHFitter: fitted with 38330 total observations, 37598 right-censored observations>
             duration col = 'CHD_DY'
                event col = 'CHD_E'
      baseline estimation = breslow
   number of observations = 38330
number of events observed = 732
   partial log-likelihood = -7503.27
         time fit was run = 2024-12-02 13:14:57 UTC

---
                                         coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                                                
AGE                                      0.10      1.10      0.01            0.09            0.11                1.09                1.12
ETHNIC_White                            -0.02      0.98      0.10           -0.23            0.18                0.80                1.20
EDUC_Some post-graduate or professional -0.27      0.76      0.13           -0.52           -0.02                0.59                0.98
EDUC_Some college or Associate Degree   -0.32      0.73      0.09           -0.49           -0.14                0.61                0.87
BMI                                      0.04      1.04      0.01            0.03            0.05                1.03                1.06
SMOKING_Past Smoker                      0.28      1.33      0.08            0.13            0.43                1.14                1.54
SMOKING_Current Smoker                   0.61      1.84      0.14            0.33            0.89                1.39                2.43
MENO                                    -0.02      0.98      0.01           -0.04           -0.01                0.96                0.99
PHYSFUN                                 -0.00      1.00      0.00           -0.01            0.00                0.99                1.00
HRTARM                                  -0.02      0.98      0.19           -0.39            0.36                0.68                1.43

                                         cmp to     z      p  -log2(p)
covariate                                                             
AGE                                        0.00 16.69 <0.005    205.41
ETHNIC_White                               0.00 -0.24   0.81      0.30
EDUC_Some post-graduate or professional    0.00 -2.14   0.03      4.96
EDUC_Some college or Associate Degree      0.00 -3.45 <0.005     10.78
BMI                                        0.00  7.95 <0.005     48.94
SMOKING_Past Smoker                        0.00  3.64 <0.005     11.82
SMOKING_Current Smoker                     0.00  4.28 <0.005     15.69
MENO                                       0.00 -3.09 <0.005      8.96
PHYSFUN                                    0.00 -1.79   0.07      3.75
HRTARM                                     0.00 -0.08   0.94      0.10
---
Concordance = 0.71
Partial AIC = 15026.53
log-likelihood ratio test = 430.46 on 10 df
-log2(p) of ll-ratio test = 284.07

Hazard ratio for CHD in Observational Study: 0.98 (95% CI: 0.68, 1.43)


<lifelines.CoxPHFitter: fitted with 40040 total observations, 39306 right-censored observations>
             duration col = 'CHD_DY'
                event col = 'CHD_E'
      baseline estimation = breslow
   number of observations = 40040
number of events observed = 734
   partial log-likelihood = -7544.74
         time fit was run = 2024-12-02 13:14:57 UTC

---
                                         coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                                                
AGE                                      0.10      1.10      0.01            0.09            0.11                1.09                1.12
ETHNIC_White                             0.00      1.00      0.10           -0.20            0.21                0.82                1.23
EDUC_Some post-graduate or professional -0.28      0.75      0.13           -0.54           -0.03                0.58                0.97
EDUC_Some college or Associate Degree   -0.26      0.77      0.09           -0.44           -0.08                0.65                0.92
BMI                                      0.04      1.04      0.01            0.03            0.05                1.03                1.05
SMOKING_Past Smoker                      0.28      1.32      0.08            0.13            0.43                1.13                1.54
SMOKING_Current Smoker                   0.57      1.76      0.14            0.28            0.85                1.33                2.34
MENO                                    -0.02      0.98      0.01           -0.04           -0.01                0.97                0.99
PHYSFUN                                 -0.00      1.00      0.00           -0.01            0.00                0.99                1.00
HRTARM                                  -0.35      0.70      0.19           -0.72            0.01                0.49                1.01

                                         cmp to     z      p  -log2(p)
covariate                                                             
AGE                                        0.00 16.85 <0.005    209.17
ETHNIC_White                               0.00  0.04   0.97      0.04
EDUC_Some post-graduate or professional    0.00 -2.21   0.03      5.21
EDUC_Some college or Associate Degree      0.00 -2.88 <0.005      7.98
BMI                                        0.00  7.82 <0.005     47.45
SMOKING_Past Smoker                        0.00  3.58 <0.005     11.51
SMOKING_Current Smoker                     0.00  3.91 <0.005     13.40
MENO                                       0.00 -2.98 <0.005      8.44
PHYSFUN                                    0.00 -1.15   0.25      1.99
HRTARM                                     0.00 -1.88   0.06      4.07
---
Concordance = 0.72
Partial AIC = 15109.49
log-likelihood ratio test = 454.32 on 10 df
-log2(p) of ll-ratio test = 300.97

Hazard ratio for CHD in Observational Study: 0.7 (95% CI: 0.49, 1.01)
(124, 12)


<lifelines.CoxPHFitter: fitted with 45756 total observations, 44929 right-censored observations>
             duration col = 'CHD_DY'
                event col = 'CHD_E'
      baseline estimation = breslow
   number of observations = 45756
number of events observed = 827
   partial log-likelihood = -8637.93
         time fit was run = 2024-12-02 13:14:58 UTC

---
                                         coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                                                
AGE                                      0.10      1.10      0.01            0.09            0.11                1.09                1.11
ETHNIC_White                            -0.00      1.00      0.10           -0.20            0.19                0.82                1.21
EDUC_Some post-graduate or professional -0.24      0.79      0.12           -0.47           -0.01                0.63                0.99
EDUC_Some college or Associate Degree   -0.25      0.78      0.08           -0.42           -0.08                0.66                0.92
BMI                                      0.04      1.04      0.01            0.03            0.05                1.03                1.05
SMOKING_Past Smoker                      0.25      1.29      0.07            0.11            0.40                1.12                1.49
SMOKING_Current Smoker                   0.56      1.74      0.14            0.29            0.83                1.33                2.28
MENO                                    -0.02      0.98      0.01           -0.03           -0.01                0.97                0.99
PHYSFUN                                 -0.00      1.00      0.00           -0.01           -0.00                0.99                1.00
HRTARM                                  -0.11      0.90      0.10           -0.31            0.09                0.74                1.09

                                         cmp to     z      p  -log2(p)
covariate                                                             
AGE                                        0.00 17.32 <0.005    220.77
ETHNIC_White                               0.00 -0.03   0.97      0.04
EDUC_Some post-graduate or professional    0.00 -2.02   0.04      4.53
EDUC_Some college or Associate Degree      0.00 -2.94 <0.005      8.27
BMI                                        0.00  7.72 <0.005     46.28
SMOKING_Past Smoker                        0.00  3.50 <0.005     11.05
SMOKING_Current Smoker                     0.00  4.05 <0.005     14.25
MENO                                       0.00 -2.88 <0.005      7.99
PHYSFUN                                    0.00 -2.09   0.04      4.79
HRTARM                                     0.00 -1.10   0.27      1.88
---
Concordance = 0.71
Partial AIC = 17295.86
log-likelihood ratio test = 458.33 on 10 df
-log2(p) of ll-ratio test = 303.81

Hazard ratio for CHD in Observational Study: 0.9 (95% CI: 0.74, 1.09)
